# MegaDescriptor-L-384
- Configuration used for training MegaDescriptor-L-384 (https://huggingface.co/BVRA/MegaDescriptor-L-384)

In [1]:
from itertools import chain
import torch
import timm
import pandas as pd
import torchvision.transforms as T
from torch.optim import SGD
from wildlife_tools.data import WildlifeDataset
from wildlife_tools.train import ArcFaceLoss, BasicTrainer
# Added to support COCO only dataset as per our model
from welfareobs.detectron.welfareobs_dataset import WelfareObsDataset
from welfareobs.utils.padded_square_transform import PaddedSquareTransform

"""
Notes on this variation of the original training notebook from the paper:

* We change the transform being used. Fundamental changes are:
  - No random augmentation
  - Padded square transform provides a square image with isotropic scaling and fills the remainder of the image with 
    the edge of the crop (if any) - see workbench-transforms.ipynb 
  - We resize to 384x384 AFTER the padded square tx to provide best resolution when cropping an image from bounding box

* We inherit the WildlifeDataset with the WelfareObsDataset which handles COCO format data instead
* Since we know we have bounding boxes in our COCOs dataset, we use `img_load='bbox'` parameter when loading images

The training that varies from the original training notebook from the paper is that I have simply repeated the training 
three times with variations to backbone and dataset. 

What we are training in this notebook are:
  1. The WoD four giraffe reidentification dataset using the SWIN baseline model
  2. The GZGC reidentification dataset using the SWIN baseline model
  3. The WoD four giraffe reidentification dataset using the MegaDescriptor model (finetuning example)

"""


# we don't use this as the WelfareObsDataset uses COCO format dataset only
# metadata = pd.read_csv('../data/metadata/combined/combined_all.csv')
# image_root = '../data/images/size-518'

# Original transform
# transform = T.Compose([
#     T.Resize(size=(384, 384)),
#     T.RandAugment(num_ops=2, magnitude=20),
#     T.ToTensor(),
#     T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
# ])

# My transform maintains aspect ratio using PaddedSquareTransform
transform = T.Compose([
    PaddedSquareTransform(fill=0, padding_mode="edge"),    
    T.Resize(
        size=(384,384),
        interpolation=T.InterpolationMode.BILINEAR,
        max_size=None,
        antialias=True
    ),
    T.ToTensor(),  # Convert a PIL Image or ndarray to tensor and scale the values 0->255 to 0.0->1.0
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # output[channel] = (input[channel] - mean[channel]) / std[channel] (this is the mapping for ImageNet RGB)
])

# Original dataset
# dataset = WildlifeDataset(
#     metadata = metadata.query('split == "train"'), 
#     root = image_root,
#     transform=transform
# )

# /project/data/reidentification/gzgc/gzgc.coco

# Dataset configuration (changed from the original paper to use inherited version of WildlifeDataset that supports COCOs JSON)
# Note that the absolute path is for the Docker image used by realtime-welfare project
dataset=WelfareObsDataset(
    root="/project/data/wod_reid",
    annotations_file="coco_train.json",
    transform=transform,
    img_load="bbox", # "bbox_mask",
    col_path="path",
    col_label="identity",
    load_label=True
)

# Backbone and loss configuration unchanged
backbone = timm.create_model('swin_large_patch4_window12_384', num_classes=0, pretrained=True)
with torch.no_grad():
    dummy_input = torch.randn(1, 3, 384, 384)
    embedding_size = backbone(dummy_input).shape[1]
objective = ArcFaceLoss(num_classes=dataset.num_classes, embedding_size=embedding_size, margin=0.5, scale=64)

# Optimizer and scheduler configuration unchanged
params = chain(backbone.parameters(), objective.parameters())
optimizer = SGD(params=params, lr=0.001, momentum=0.9)
min_lr = optimizer.defaults.get("lr") * 1e-3
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=min_lr)

# Setup training
trainer = BasicTrainer(
    dataset=dataset,
    model=backbone,
    objective=objective,
    optimizer=optimizer,
    scheduler=scheduler,
    batch_size=32,  # Original was 16, doubled to reduce training time on our GPU
    accumulation_steps=8,
    num_workers=2,
    epochs=100,
    device='cuda',
)

trainer.train()

# added this so we can use the model for inference
trainer.save("/project/data/md-validation")

Epoch 99: 100%|███████████████████████████████████████████████████| 163/163 [03:54<00:00,  1.44s/it]


In [4]:
# Variation on the above code using the pretrained foundation model as the starting point (fine tuning)
from itertools import chain
import torch
import timm
import pandas as pd
import torchvision.transforms as T
from torch.optim import SGD
from wildlife_tools.data import WildlifeDataset
from wildlife_tools.train import ArcFaceLoss, BasicTrainer
from welfareobs.detectron.welfareobs_dataset import WelfareObsDataset


# Original transform
transform = T.Compose([
    PaddedSquareTransform(fill=0, padding_mode="edge"),    
    T.Resize(
        size=(384,384),
        interpolation=T.InterpolationMode.BILINEAR,
        max_size=None,
        antialias=True
    ),
    T.ToTensor(),  # Convert a PIL Image or ndarray to tensor and scale the values 0->255 to 0.0->1.0
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # output[channel] = (input[channel] - mean[channel]) / std[channel] (this is the mapping for ImageNet RGB)
])

# Here we have a use the WoD dataset to finetune the pretrained MD model
dataset = WelfareObsDataset(
    root="/project/data/wod_reid",
    annotations_file="coco_train.json",
    transform=transform,
    img_load="bbox", # "bbox_mask",
    col_path="path",
    col_label="identity",
    load_label=True
)

# Backbone and loss configuration unchanged
backbone = timm.create_model("hf-hub:BVRA/wildlife-mega-L-384", pretrained=True)
with torch.no_grad():
    dummy_input = torch.randn(1, 3, 384, 384)
    embedding_size = backbone(dummy_input).shape[1]
objective = ArcFaceLoss(num_classes=dataset.num_classes, embedding_size=embedding_size, margin=0.5, scale=64)

# Optimizer and scheduler configuration unchanged
params = chain(backbone.parameters(), objective.parameters())
optimizer = SGD(params=params, lr=0.001, momentum=0.9)
min_lr = optimizer.defaults.get("lr") * 1e-3
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=min_lr)

# Setup training
trainer = BasicTrainer(
    dataset=dataset,
    model=backbone,
    objective=objective,
    optimizer=optimizer,
    scheduler=scheduler,
    batch_size=32,  # Original was 16, doubled to reduce training time on our GPU
    accumulation_steps=8,
    num_workers=2,
    epochs=100,
    device='cuda',
)

trainer.train()
trainer.save("/project/data/md-validation-finetuning")


Epoch 99: 100%|███████████████████████████████████████████████████| 163/163 [03:55<00:00,  1.44s/it]
